# KeyFrame: Piano Key Detection from Video

**Name:** Michelle Adu-Asah

**Course:** ITAI 1378 Computer Vision

**Project:** Final Project: KeyFrame

**Tier:** Tier 1

**Date:** 10 October 2026


KeyFrame is a computer vision project that watches a short video of a piano and figures out which keys were played.
The goal is to detect a finger on the keyboard and turn its position into note names like G2 or C3.

In this notebook, I load my piano videos, run the detection, and check how accurate it is.

## Step 1: Set Up the Note Detector

This cell contains the main function, `read_notes`. It finds the first key, tracks the fingertip, and converts its position into a note.
Run this cell first.

In [40]:
import cv2, numpy as np
from google.colab import files

# key_frac = width of one white key as a share of the video width
# 0.048 for my first recordings, about 0.043 for the new ones (camera farther back)
def read_notes(path, key_frac=0.048, min_sec=0.2):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    keys = []
    while True:
        ok, f = cap.read()
        if not ok: break
        H, W = f.shape[:2]
        hsv = cv2.cvtColor(f, cv2.COLOR_BGR2HSV)
        # 1) left edge of first key: first bright stretch after the dark keyboard edge
        row = hsv[int(H * 0.85), :, 2]
        left, seen_dark = 0, False
        for x in range(W - 35):
            if row[x] < 100: seen_dark = True
            if seen_dark and (row[x:x + 35] > 150).all():
                left = x; break
        # 2) fingertip near the bottom
        m = cv2.inRange(hsv, (0, 90, 40), (13, 255, 215))
        m[:int(H * 0.75)] = 0
        m[:, :left] = 0
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        key = None
        if cnts:
            c = max(cnts, key=cv2.contourArea)
            if cv2.contourArea(c) > 250:
                tip_x = c[c[:, 0, 1].argmin()][0][0]
                key = int((tip_x - left) // (W * key_frac)) + 1   # white key number
        keys.append(key)
    # 3) a note = same key held for min_sec
    notes, i, need = [], 0, int(min_sec * fps)
    while i < len(keys):
        j = i
        while j + 1 < len(keys) and keys[j + 1] == keys[i]: j += 1
        if keys[i] and j - i + 1 >= need:
            n = keys[i]
            name = "CDEFGAB"[(n - 1) % 7] + str((n - 1) // 7 + 1)
            if not notes or notes[-1] != name: notes.append(name)
        i = j + 1
    return notes

## Step 2: Try It on a Video

Upload a video using the Files panel, and the program will tell you which notes it thinks were played.
This is the demo of the system working on real input.

In [36]:
import os

print("Welcome to KeyFrame!")
print("Upload your piano videos using the Files panel (folder icon on the left).")

done = set()

while True:
    answer = input("\nPress Enter when your video is uploaded (or type q to quit): ").strip().lower()
    if answer in ("q", "quit", "no"):
        print("Thanks for using KeyFrame!")
        break

    videos = sorted(f for f in os.listdir("/content")
                    if f.lower().startswith("example") and f.lower().endswith((".mov", ".mp4")))
    new_videos = [v for v in videos if v not in done]

    if not new_videos:
        print("I don't see any new videos yet. Check the Files panel and try again.")
        continue

    for name in new_videos:
        print(f"\nListening to {name} ...")
        notes = read_notes(os.path.join("/content", name))

        if not notes:
            print("I couldn't find any notes in this video.")
        elif len(notes) == 1:
            print("You played this note:", notes[0])
        else:
            print("You played these notes:", " -> ".join(notes))

        done.add(name)

Welcome to KeyFrame!
Upload your piano videos using the Files panel (folder icon on the left).

Press Enter when your video is uploaded (or type q to quit): 

Listening to Example 1.mov ...
You played these notes: B2 -> C3

Listening to Example 2.mov ...
You played these notes: C1 -> E1 -> A1 -> D2 -> E2

Listening to Example 3.mov ...
You played these notes: G2 -> A2 -> G2 -> E2 -> D2 -> E2

Listening to Example 4.mov ...
You played these notes: C2 -> D2 -> E2 -> F2 -> G2 -> A2

Listening to Example 5.mov ...
You played these notes: A2 -> G2 -> F2 -> E2 -> D2 -> E2 -> D2 -> C2

Listening to Example 6.mov ...
You played these notes: F2 -> D2 -> A2

Listening to Example 7.mov ...
You played these notes: G2 -> C3

Listening to Example 8.mov ...
You played these notes: G2 -> D2

Press Enter when your video is uploaded (or type q to quit): q
Thanks for using KeyFrame!


## Step 3: Baseline Results on My Original Clips

Here I compare what the program predicted with the notes I actually played in my first 8 recordings.
This shows how well the system works before any improvements, and where it makes mistakes.

In [37]:
import os

truth = {
    "Example 1.mov": "B2 C3",
    "Example 2.mov": "C1 E1 A1 E2",
    "Example 3.mov": "G2 D2",
    "Example 4.mov": "C2 D2 E2 F2 G2 A2",
    "Example 5.mov": "A2 G2 F2 E2 D2 C2",
    "Example 6.mov": "F2 D2 A2",
    "Example 7.mov": "G2 C3",
    "Example 8.mov": "G2 D2",
}

# ONLY process files that are explicitly listed in the truth dictionary above
videos = sorted(f for f in os.listdir("/content") if f in truth)

correct = 0
total = 0

for name in videos:
    pred = " ".join(read_notes(os.path.join("/content", name)))
    ok = pred == truth[name]
    correct += ok
    total += 1
    print(f"{name[:25]}\n  Truth:     {truth[name]}\n  Predicted: {pred}\n  Exact match: {ok}\n")

if total:
    print(f"Exact clips: {correct}/{total}")

Example 1.mov
  Truth:     B2 C3
  Predicted: B2 C3
  Exact match: True

Example 2.mov
  Truth:     C1 E1 A1 E2
  Predicted: C1 E1 A1 D2 E2
  Exact match: False

Example 3.mov
  Truth:     G2 D2
  Predicted: G2 A2 G2 E2 D2 E2
  Exact match: False

Example 4.mov
  Truth:     C2 D2 E2 F2 G2 A2
  Predicted: C2 D2 E2 F2 G2 A2
  Exact match: True

Example 5.mov
  Truth:     A2 G2 F2 E2 D2 C2
  Predicted: A2 G2 F2 E2 D2 E2 D2 C2
  Exact match: False

Example 6.mov
  Truth:     F2 D2 A2
  Predicted: F2 D2 A2
  Exact match: True

Example 7.mov
  Truth:     G2 C3
  Predicted: G2 C3
  Exact match: True

Example 8.mov
  Truth:     G2 D2
  Predicted: G2 D2
  Exact match: True

Exact clips: 5/8


## Step 4: Test on New Clips

These are new videos the system has not seen before. I write down the notes before recording, then the program predicts them.
This gives an honest measure of accuracy.

In [39]:
import os

# Find new test videos (file names starting with "test")
videos = sorted(f for f in os.listdir("/content")
                if f.lower().startswith("test") and f.lower().endswith((".mov", ".mp4")))

if not videos:
    print("No test videos found. Upload to the Files panel and name them test1.mov, test2.mov, ...")

# Count of correct clips and stored results
correct = 0
results = []

# Loop through each test video
for name in videos:

    # Ask for the real notes before showing the prediction
    truth = input(f"What notes were played in {name}? (example: G2 C3): ").strip().upper()

    # Predict the notes from the video
    pred = " ".join(read_notes(os.path.join("/content", name), key_frac=0.043))

    # Compare prediction with the real notes
    ok = pred == truth
    correct += ok
    results.append((name, truth, pred, ok))

    print(f"  Predicted: {pred}")
    print(f"  Exact match: {ok}\n")

# Summary table of all test clips
print("SUMMARY")
for name, truth, pred, ok in results:
    print(f"{name} | played: {truth} | predicted: {pred} | {'OK' if ok else 'MISMATCH'}")

# Final score
if results:
    print(f"\nExact clips: {correct}/{len(results)}")

What notes were played in Test 1.mov? (example: G2 C3): G1 F1 C1
  Predicted: F1 G1 F1 C1
  Exact match: False

What notes were played in Test 2.mov? (example: G2 C3): A3 C4 F3
  Predicted: B3 G3 B3 E3
  Exact match: False

What notes were played in Test 3.mov? (example: G2 C3): B2 G3
  Predicted: B2 G3
  Exact match: True

SUMMARY
Test 1.mov | played: G1 F1 C1 | predicted: F1 G1 F1 C1 | MISMATCH
Test 2.mov | played: A3 C4 F3 | predicted: B3 G3 B3 E3 | MISMATCH
Test 3.mov | played: B2 G3 | predicted: B2 G3 | OK

Exact clips: 1/3
